# TOV background 

In [ ]:
import numpy as np
import cmath as cm
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq

pi = np.pi
Msun = 1.47664 # Solar mass in geometric units (km)
K = 100 * Msun**2
#K = 1.0
n = 1
Gamma = (1 + 1/n)  # Polytropic index


def rho_from_p(p, K):
    rho_b = (np.maximum(p, 1e-13) / K)**0.5
    return rho_b + p

def cs2_from_p(p, K):
    rho_b = (np.maximum(p, 1e-13) / K)**0.5
    return 1.0 / (rho_b/(2*p) + 1.0)



def tov_rhs(r, y, K):
    m, p, nu = y
    
    if p <= 0:
        return [0.0, 0.0, 0.0]
    
    rho = rho_from_p(p, K)
    
    dmdr = 4*pi*r**2*rho
    
    dpdr = -(rho + p)*(m + 4*pi*r**3*p)/(r*(r-2*m))
    
    dnudr = 2*(m + 4*pi*r**3*p)/(r*(r-2*m))
    
    return [dmdr, dpdr, dnudr]

def surface_event(r, y, K):
    return y[1]

surface_event.terminal = True
surface_event.direction = -1



def solve_tov(p_c, K, r0=1e-6, r_max=50.0):
    
    rho_c = rho_from_p(p_c, K)
    
    m0 = (4/3)*pi*rho_c*r0**3
    nu0 = 0.0
    
    y0 = [m0, p_c, nu0]
    
    sol = solve_ivp(
        lambda r, y: tov_rhs(r, y, K),
        (r0, r_max),
        y0,
        events=lambda r,y: surface_event(r,y,K),
        rtol=1e-9,
        atol=1e-12
    )
    
    if sol.t_events[0].size == 0:
        raise RuntimeError("Surface not found.")
    
    R = sol.t_events[0][0]
    M = sol.y_events[0][0][0]
    nu_surface = sol.y_events[0][0][2]
    
    mask = sol.t <= R
    r_vals = sol.t[mask]
    m_vals = sol.y[0][mask]
    p_vals = sol.y[1][mask]
    nu_vals = sol.y[2][mask]
    
    # Force last point to be exactly R
    r_vals[-1] = R
    m_vals[-1] = M
    p_vals[-1] = 0.0
    nu_vals[-1] = nu_surface
    
    # Normalize metric
    nu_shift = np.log(1 - 2*M/R) - nu_surface
    nu_vals = nu_vals + nu_shift
    
    return r_vals, m_vals, p_vals, nu_vals, M, R
"""
def find_pc_for_compactness(C_target, K):
    
    def f(p_c):
        r, m, p, nu, M, R = solve_tov(p_c, K)
        return M/R - C_target
    
    return brentq(f, 1e-4, 10.0)

"""
def find_pc_for_compactness(C_target, K):
    
    def f(p_c):
        r, m, p, nu, M, R = solve_tov(p_c, K)
        return M/R - C_target
    
    # Scan log-space for bracket
    p_scan = np.logspace(-6, 2, 50)
    
    f_vals = []
    for p in p_scan:
        try:
            f_vals.append(f(p))
        except:
            f_vals.append(np.nan)
    
    for i in range(len(p_scan)-1):
        if np.isnan(f_vals[i]) or np.isnan(f_vals[i+1]):
            continue
        if f_vals[i]*f_vals[i+1] < 0:
            return brentq(f, p_scan[i], p_scan[i+1])
    
    raise ValueError("Target compactness not bracketed.")

def build_background_for_C(C_target, K):
    
    p_c = find_pc_for_compactness(C_target, K)
    
    r, m, p, nu, M, R = solve_tov(p_c, K)
    
    eps = rho_from_p(p, K)
    cs2 = cs2_from_p(p, K)
    
    lam = np.log(1/(1 - 2*m/r))
    
    # Derivatives
    p_spline = CubicSpline(r, p)
    eps_spline = CubicSpline(r, eps)
    nu_spline = CubicSpline(r, nu)
    
    p_p = p_spline.derivative()(r)
    eps_p = eps_spline.derivative()(r)
    nu_p = nu_spline.derivative()(r)
    
    bg_dict = {
        C_target: {
            "M": M,
            "R": R,
            "r": r,
            "m": m,
            "p": p,
            "eps": eps,
            "nu": nu,
            "lam": lam,
            "nu_p": nu_p,
            "p_p": p_p,
            "eps_p": eps_p,
            "cs2": cs2
        }
    }
    
    return bg_dict



In [ ]:
def build_background_for_C(C_target, K):
    
    p_c = find_pc_for_compactness(C_target, K)
    
    r, m, p, nu, M, R = solve_tov(p_c, K)
    
    eps = rho_from_p(p, K)
    cs2 = cs2_from_p(p, K)
    
    lam = np.log(1/(1 - 2*m/r))
    
    # Smooth derivatives via splines
    p_spline = CubicSpline(r, p)
    eps_spline = CubicSpline(r, eps)
    nu_spline = CubicSpline(r, nu)
    
    p_p = p_spline.derivative()(r)
    eps_p = eps_spline.derivative()(r)
    nu_p = nu_spline.derivative()(r)
    
    return {
        "M": M,
        "R": R,
        "C": M/R,
        "r": r,
        "m": m,
        "p": p,
        "eps": eps,
        "nu": nu,
        "lam": lam,
        "nu_p": nu_p,
        "p_p": p_p,
        "eps_p": eps_p,
        "cs2": cs2
    }

def build_backgrounds_for_C_array(C_array, K):
    
    bg_dict = {}
    
    for C in C_array:
        print(f"Building star for C = {C:.6f}")
        bg_dict[C] = build_background_for_C(C, K)
    
    return bg_dict



K = 100*(Msun)**2

#C_values = np.array([0.12, 0.14, 0.16])

C_values = np.round(np.arange(0.10, 0.24, 0.01), 2)

backgrounds = build_backgrounds_for_C_array(C_values, K)
print(" C        M        R")
print("-"*30)

for C in backgrounds:
    M = backgrounds[C]["M"]
    R = backgrounds[C]["R"]
    print(f"{C:.6f}  {M:.6f}  {R:.6f}")


import matplotlib.pyplot as plt

M_vals = [backgrounds[C]["M"] for C in backgrounds]
R_vals = [backgrounds[C]["R"] for C in backgrounds]

plt.plot(R_vals, M_vals, 'o-')
plt.xlabel("R")
plt.ylabel("M")
plt.title("Mass-Radius curve")
plt.show()

In [ ]:
def scan_compactness(K):
    p_vals = np.logspace(-6, 2, 40)
    print(" p_c        C")
    print("--------------------")
    for p in p_vals:
        try:
            r, m, p_arr, nu, M, R = solve_tov(p, K)
            print(f"{p:10.4e}  {M/R:.6f}")
        except:
            print(f"{p:10.4e}  FAILED")

#scan_compactness(K)

import numpy as np
import matplotlib.pyplot as plt

def compute_compactness_curve(K, p_min=1e-6, p_max=1e3, npts=100):
    
    p_vals = np.logspace(np.log10(p_min), np.log10(p_max), npts)
    C_vals = []
    
    for p in p_vals:
        try:
            r, m, p_arr, nu, M, R = solve_tov(p, K)
            C_vals.append(M/R)
        except:
            C_vals.append(np.nan)
    
    return p_vals, np.array(C_vals)

# Compute
p_vals, C_vals = compute_compactness_curve(K)

# Plot
imax = np.nanargmax(C_vals)
print("Maximum C =", C_vals[imax])
print("At p_c =", p_vals[imax])

plt.figure(figsize=(6,4))
plt.plot(p_vals, C_vals)
plt.scatter(p_vals[imax], C_vals[imax], color='red')
plt.xscale("log")
plt.xlabel("Central Pressure $p_c$")
plt.ylabel("Compactness $C$")
plt.title("Compactness Curve (Maximum Marked)")
plt.grid(True)
plt.show()

def compute_MR_curve(K, p_min=1e-6, p_max=1e3, npts=100):
    
    p_vals = np.logspace(np.log10(p_min), np.log10(p_max), npts)
    M_vals = []
    R_vals = []
    
    for p in p_vals:
        try:
            r, m, p_arr, nu, M, R = solve_tov(p, K)
            M_vals.append(M)
            R_vals.append(R)
        except:
            M_vals.append(np.nan)
            R_vals.append(np.nan)
    
    return np.array(M_vals), np.array(R_vals)

M_vals, R_vals = compute_MR_curve(K)

plt.figure(figsize=(6,4))
plt.plot(R_vals, M_vals)
plt.xlabel("R")
plt.ylabel("M")
plt.title("Mass-Radius Curve")
plt.grid(True)
plt.show()

In [ ]:
K = 100*(1.47664)**2   # same as Mathematica

bg = backgrounds#build_background_for_C(0.1460474, K)

C = list(bg.keys())[0]
R = bg[C]["R"]
M = bg[C]["M"]

print("M =", M)
print("R =", R)
print("C =", M/R)

In [ ]:

C = 0.2

M = backgrounds[C]['M']
R = backgrounds[C]['R']
nu_in = backgrounds[C]['nu']
r_in = backgrounds[C]['r']
m = backgrounds[C]['m']

def nu_out(r):
    return np.log(1 - 2*M/r)


exp_nu_in = np.exp(nu_in)
r_out = np.linspace(R, 5*R, 200)
exp_nu_out = np.exp(nu_out(r_out))


plt.figure(figsize=(7,5))
plt.plot(r_in, exp_nu_in, label=r'$e^{\nu(r)}_{\mathrm{inside}}$', color='tab:blue')
plt.plot(r_out, exp_nu_out, label=r'$e^{\nu(r)}_{\mathrm{outside}}$', color='tab:orange', linestyle='--')

plt.axvline(R, color='k', linestyle=':', label=r'$r=R$ (surface)')
plt.xlabel(r'$r$ [km]')
plt.ylabel(r'$e^{\nu(r)}$')
plt.title('Continuity of $e^{\\nu(r)}$ at the stellar surface')
plt.legend()
plt.grid(True, ls=':')
plt.show()


elam_inside = 1 - 2 * m / r_in

r_out = np.linspace(R, 5 * R, 200)
elam_outside = 1 - 2 * M / r_out

plt.figure(figsize=(6,5))
plt.plot(r_in, elam_inside, label=r"$e^{-\lambda(r)}_{\rm inside}$", lw=2)
plt.plot(r_out, elam_outside, '--', label=r"$e^{-\lambda(r)}_{\rm outside}$", lw=2)
plt.axvline(R, color='k', ls=':', label=r"$r = R$ (surface)")
plt.xlabel(r"$r$ [km]")
plt.ylabel(r"$e^{-\lambda(r)}$")
plt.title(r"Continuity of $e^{-\lambda(r)}$ at the stellar surface")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# Perturbation helpers

In [ ]:
def make_bg_tables_from_profile(profile, r_grid):
    """
    profile = one entry from backgrounds[C]
    r_grid  = interpolation grid (np.linspace(eps, R, N))
    """

    r_prof = profile['r']
    
    bg_tab = {
        'r': r_grid,
        'm': np.interp(r_grid, r_prof, profile['m']),
        'p': np.interp(r_grid, r_prof, profile['p']),
        'eps': np.interp(r_grid, r_prof, profile['eps']),
        'nu': np.interp(r_grid, r_prof, profile['nu']),
        'lam': np.interp(r_grid, r_prof, profile['lam']),
        'nu_p': np.interp(r_grid, r_prof, profile['nu_p']),
        'p_prime': np.interp(r_grid, r_prof, profile['p_p']),
        'eps_prime': np.interp(r_grid, r_prof, profile['eps_p']),
        'cs2': np.interp(r_grid, r_prof, profile['cs2']),
    }

    # lam_p computed from definition instead of spline
    bg_tab['lam_p'] = (
        2*(bg_tab['m'] + 4*np.pi*r_grid**3*bg_tab['p'])
        / (r_grid*(r_grid - 2*bg_tab['m']))
    )

    return bg_tab

def pack_bg_quantities(bg_tab):
    return np.vstack([
        bg_tab['m'],
        bg_tab['p'],
        bg_tab['eps'],
        bg_tab['nu'],
        bg_tab['lam'],
        bg_tab['lam_p'],
        bg_tab['nu_p'],
        bg_tab['p_prime'],
        bg_tab['eps_prime'],
        bg_tab['cs2'],
    ])

profile = backgrounds[0.2]

R = profile['R']
r_start = 1e-6 
r_tab = np.linspace(r_start, R, 4000)

bg_tab = make_bg_tables_from_profile(profile, r_tab)
bg_packed = pack_bg_quantities(bg_tab)


class ComplexSpline:
    def __init__(self, x, y, k=3):
        self.real_spline = CubicSpline(x, np.real(y))
        self.imag_spline = CubicSpline(x, np.imag(y))

    def __call__(self, x):
        return self.real_spline(x) + 1j * self.imag_spline(x)

    def derivative(self, n=1):
        der_real = self.real_spline.derivative(n)
        der_imag = self.imag_spline.derivative(n)
        return lambda x: der_real(x) + 1j * der_imag(x)


class SourceTensor:
  def S01(self, r, omega): return 0.0
  def S01p(self, r, omega): return 0.0
  def S00(self, r, omega): return 0.0
  def S0A(self, r, omega): return 0.0
  def SZ(self, r, omega): return 0.0
  def S0(self, r, omega): return 0.0
  def S0p(self, r, omega): return 0.0
  def S1(self, r, omega): return 0.0
  def S1p(self, r, omega): return 0.0
  def SOm(self, r, omega): return 0.0

def make_source(source, r_grid, omega):

    return {
        'S01':  np.asarray([source.S01(r,  omega) for r in r_grid]),
        'S01p': np.asarray([source.S01p(r, omega) for r in r_grid]),
        'S00':  np.asarray([source.S00(r,  omega) for r in r_grid]),
        'S0A':  np.asarray([source.S0A(r,  omega) for r in r_grid]),
        'SZ':   np.asarray([source.SZ(r,   omega) for r in r_grid]),
        'S0':   np.asarray([source.S0(r,   omega) for r in r_grid]),
        'S0p':  np.asarray([source.S0p(r,  omega) for r in r_grid]),
        'S1':   np.asarray([source.S1(r,   omega) for r in r_grid]),
        'S1p':  np.asarray([source.S1p(r,  omega) for r in r_grid]),
        'SOm':  np.asarray([source.SOm(r,  omega) for r in r_grid]),
    }

def pack_source(source_tables):
    return np.vstack([source_tables[k] for k in ('S01','S01p','S00','S0A','SZ','S0','S0p','S1','S1p','SOm')])



from numba import njit
"""
@njit()
def interp(r, r_grid, f_grid):
    i = np.searchsorted(r_grid, r) - 1
    if i < 0:
        i = 0
    if i >= r_grid.size - 1:
        i = r_grid.size - 2

    r0 = r_grid[i]
    r1 = r_grid[i+1]
    f0 = f_grid[i]
    f1 = f_grid[i+1]

    return f0 + (f1 - f0) * (r - r0) / (r1 - r0)
"""
@njit
def interp(x, x_grid, y_grid):

    n = x_grid.size

    # detect direction
    increasing = x_grid[0] < x_grid[-1]

    if increasing:
        i = np.searchsorted(x_grid, x) - 1
    else:
        # search on reversed logic
        i = np.searchsorted(x_grid[::-1], x) - 1
        i = n - 2 - i

    if i < 0:
        i = 0
    if i > n - 2:
        i = n - 2

    x0 = x_grid[i]
    x1 = x_grid[i+1]
    y0 = y_grid[i]
    y1 = y_grid[i+1]

    return y0 + (y1 - y0) * (x - x0) / (x1 - x0)
       
  
def init_conds_even_perfect(r, om, profile, sign=+1):

    p0 = profile['p'][0]
    eps0 = profile['eps'][0]
    nu0 = profile['nu'][0]

    Whinit = 1.0
    Kinit = sign * 1 #(eps0 + p0)

    pi = np.pi

    Xinit = (1/6)*np.exp(-nu0/2)*(p0+eps0)*(-3*om**2 *Whinit+ np.exp(nu0)*(3*Kinit + 8*pi*Whinit*(3*p0+eps0)))

    H1init = (2/3)*(Kinit + 4*pi*Whinit*(p0+eps0))

    return np.array([
        Xinit.real, Xinit.imag,
        Whinit.real, Whinit.imag,
        H1init.real, H1init.imag,
        Kinit.real, Kinit.imag
    ])

print(init_conds_even_perfect(r_tab[0], 0.5, bg_tab))


#Old init conds: [3.85586324e-14 0.00000000e+00 2.13262475e-19 0.00000000e+00 5.17056176e-37 0.00000000e+00 1.00000000e-12 0.00000000e+

In [1]:
import numpy as np

rtest = np.linspace(1e-6, 12, 4000)



In [2]:
rtest[0:10]

array([1.00000000e-06, 3.00174994e-03, 6.00249987e-03, 9.00324981e-03,
       1.20039997e-02, 1.50047497e-02, 1.80054996e-02, 2.10062496e-02,
       2.40069995e-02, 2.70077494e-02])